In [6]:
import pandas as pd
import numpy as np

# Set your grid density here. 
# 1.0 = 1 degree (~111 km distance between points)
# 0.1 = High resolution (~11 km distance)
RESOLUTION = 0.5 

print("Generating high-speed tectonic grid...")

# Comprehensive Tectonic Segments for the Indian Subcontinent & SE Asia
segments = [
    {"name": "Chile-Peru Trench", "lat_range": (-55, -5), "lon_fixed": -72},
    {"name": "Cascadia Subduction", "lat_range": (40, 50), "lon_fixed": -125},
    {"name": "Japan Trench", "lat_range": (30, 45), "lon_fixed": 142},

    {"name": "Aleutian Trench", "lat_fixed": 52, "lon_range": (-180, -130)},
    {"name": "Kuril-Kamchatka", "lat_range": (40, 55), "lon_fixed": 150},
   
    {"name": "Izu-Bonin-Mariana", "lat_range": (15, 30), "lon_fixed": 145},
    {"name": "Philippine Trench", "lat_range": (5, 20), "lon_fixed": 126},
    {"name": "Java-Sumatra Trench", "lat_range": (-10, 5), "lon_fixed": 100},
    {"name": "Tonga-Kermadec", "lat_range": (-35, -15), "lon_fixed": -175},
    {"name": "Himalayan Thrust", "lat_fixed": 28, "lon_range": (70, 100)},
    {"name": "Mediterranean-Hellenic", "lat_fixed": 35, "lon_range": (15, 30)},
    {"name": "Andaman-Nicobar Trench", "lat_range": (5, 15), "lon_fixed": 93},
    
    # --- NEW INDIAN & SE ASIAN SECTIONS ---
    # 1. The Andaman-Sumatra-Java Trench (The 2004 Tsunami Source)
    # This covers the arc from the Bay of Bengal down to Indonesia
    {"name": "Sumatra-Java Trench", "lat_range": (-10, 5), "lon_fixed": 102},
    {"name": "Andaman-Nicobar Segment", "lat_range": (6, 15), "lon_fixed": 93},
    
    # 2. Himalayan Main Frontal Thrust (Continental Collision)
    # Critical for Nepal, North India, and Bhutan
    {"name": "Indo-Gangetic Plain / Himalayan Front", "lat_range": (27,28), "lon_range": (75, 95)},
    
    # 3. Kachchh Rift & Makran Subduction (The 2001 Gujarat Trigger)
    # This covers the Gujarat coast and the active subduction off Pakistan
    {"name": "Kachchh-Gujarat Rift", "lat_fixed": 23, "lon_range": (69, 72)},
    {"name": "Makran Subduction Zone", "lat_range": (24, 25), "lon_range": (60, 70)},
    
    # 4. Indo-Burmese Arc (North East India / Myanmar)
    {"name": "Indo-Burmese Arc", "lat_range": (20, 27), "lon_fixed": 94},
    
    # 5. Central Indian Ridge (Oceanic segments influencing South India/Sri Lanka)
    {"name": "Ninety East Ridge", "lat_range": (-10, 10), "lon_fixed": 90}
]

dfs = []

# Loop only through the segment definitions (very fast)
for seg in segments:
    if "lat_range" in seg:
        # np.arange generates the entire list of coordinates instantly in C-memory
        # We add RESOLUTION/2 to the end to ensure the final coordinate is included
        lats = np.arange(seg["lat_range"][0], seg["lat_range"][1] + (RESOLUTION/2), RESOLUTION)
        lon_fixed = seg.get("lon_fixed")
        lon_range = seg.get("lon_range")

        if lon_fixed is not None:
            lon_val = lon_fixed
            # Create a vectorized DataFrame for this entire segment at once
            df_seg = pd.DataFrame({
                "zone": seg["name"],
                "latitude": np.round(lats, 4),
                "longitude": lon_val
            })
        elif lon_range is not None:
            lons = np.arange(lon_range[0], lon_range[1] + (RESOLUTION/2), RESOLUTION)
            lat_grid, lon_grid = np.meshgrid(lats, lons, indexing="ij")
            df_seg = pd.DataFrame({
                "zone": seg["name"],
                "latitude": np.round(lat_grid.ravel(), 4),
                "longitude": np.round(lon_grid.ravel(), 4)
            })
        else:
            raise ValueError(f"Segment is missing lon_fixed/lon_range: {seg['name']}")
    else:
        lons = np.arange(seg["lon_range"][0], seg["lon_range"][1] + (RESOLUTION/2), RESOLUTION)
        
        df_seg = pd.DataFrame({
            "zone": seg["name"],
            "latitude": seg["lat_fixed"],
            "longitude": np.round(lons, 4)
        })
        
    dfs.append(df_seg)

# pd.concat merges all segment blocks into one final DataFrame instantly
subduction_df = pd.concat(dfs, ignore_index=True)

# Export to CSV
file_name = 'subduction_zones.csv'
subduction_df.to_csv(file_name, index=False)

print(f"✅ Success! Generated {len(subduction_df)} grid points at {RESOLUTION}° resolution.")
print(f"File saved as '{file_name}'.")

Generating high-speed tectonic grid...
✅ Success! Generated 831 grid points at 0.5° resolution.
File saved as 'subduction_zones.csv'.


Z-Score Threshold,Rarity (Top %),Expected Occurrences in 15 Years,Real-World Meaning
Z>2.0,2.28%,≈250 Windows,An alert triggers roughly every 3 weeks.
Z>2.33 (P99),1.00%,≈109 Windows,An alert triggers roughly once every 2 months.
Z>2.5,0.62%,≈68 Windows,An alert triggers roughly 4 times a year.
Z>3.0,0.13%,≈14 Windows,An alert triggers roughly once a year.

In [23]:
con = duckdb.connect('mega_quake_unified.db')
con.execute("CREATE OR REPLACE TABLE subduction_zones AS SELECT * FROM subduction_df")
res_df = con.execute("SELECT * FROM subduction_zones").df()
print(res_df.head())
con.close()


                zone  latitude  longitude
0  Chile-Peru Trench     -55.0      -72.0
1  Chile-Peru Trench     -54.5      -72.0
2  Chile-Peru Trench     -54.0      -72.0
3  Chile-Peru Trench     -53.5      -72.0
4  Chile-Peru Trench     -53.0      -72.0
